# Збір реєстру резидентів Дія.City

Ноутбук виконувався в Google Colab (08.11.2026). Він обходить усі сторінки реєстру https://city.diia.gov.ua/registry/resident, розкриває списки рішень і зберігає Excel-файл з аркушами «Реєстр» і «Рішення».

На сайті немає окремої колонки з датою втрати статусу. Для компаній зі статусом «Не резидент» дата береться як дата останнього рішення Мінцифри зі списку рішень.

In [ ]:
!pip install -q playwright
!playwright install --with-deps chromium

In [ ]:
import re
import pandas as pd
from playwright.async_api import async_playwright

URL = "https://city.diia.gov.ua/registry/resident"
MAX_PAGES = None       # для тесту можна поставити 2
DEC_RE = re.compile(r"№\s*([^\s,]+),\s*від\s*(\d{2}\.\d{2}\.\d{4})")

async def parse_table(page):
    rows, decs = [], []
    for tr in await page.query_selector_all("table tbody tr"):
        tds = await tr.query_selector_all("td")
        if len(tds) < 9:
            continue
        txt = [(await t.inner_text()).strip() for t in tds]
        code, cell = txt[3], tds[8]

        toggle = await cell.query_selector("text=Переглянути всі рішення")
        if toggle:
            try:
                await toggle.click(timeout=10000)
                await page.wait_for_timeout(300)
            except Exception:
                pass

        cell_text = await cell.inner_text()
        hrefs = []
        for a in await cell.query_selector_all("a"):
            t = (await a.inner_text()).lower()
            h = await a.get_attribute("href")
            if "завантажити" in t and h:
                hrefs.append(h)
        found = DEC_RE.findall(cell_text)
        if found:
            for i, (num, d) in enumerate(found):
                decs.append({"ЄДРПОУ": code, "Номер рішення": num, "Дата рішення": d,
                             "Посилання": hrefs[i] if i < len(hrefs) else None})
        elif hrefs:
            decs.append({"ЄДРПОУ": code, "Номер рішення": None, "Дата рішення": None,
                         "Посилання": hrefs[0]})

        rows.append({
            "№ з/п": txt[0], "Орг.-правова форма": txt[1], "Назва компанії": txt[2],
            "ЄДРПОУ": code, "Дата набуття статусу": txt[4],
            "Статус на сайті": txt[5], "Стартап": txt[6], "Відповідність доходу": txt[7],
        })
    return rows, decs

async def wait_loader(page):
    try:
        await page.wait_for_selector("#website-loader", state="hidden", timeout=60000)
    except Exception:
        pass

def save(rows, decs, name):
    df = pd.DataFrame(rows).drop_duplicates(subset=["№ з/п"])
    dd = pd.DataFrame(decs).drop_duplicates()
    df["Дата набуття статусу"] = pd.to_datetime(df["Дата набуття статусу"], format="%d.%m.%Y", errors="coerce")
    df["Стартап"] = df["Стартап"].eq("Стартап").map({True: "Так", False: "Ні"})
    if not dd.empty:
        dd["Дата рішення"] = pd.to_datetime(dd["Дата рішення"], format="%d.%m.%Y", errors="coerce")
        g = dd.groupby("ЄДРПОУ")["Дата рішення"].agg(["count", "max"]).reset_index()
        g.columns = ["ЄДРПОУ", "Кількість рішень", "Дата останнього рішення"]
        df = df.merge(g, on="ЄДРПОУ", how="left")
    else:
        df["Кількість рішень"], df["Дата останнього рішення"] = 0, pd.NaT
    # кандидат на дату втрати: останнє рішення для «Не резидент», якщо воно пізніше за набуття
    mask = (df["Статус на сайті"] == "Не резидент") & (df["Дата останнього рішення"] > df["Дата набуття статусу"])
    df["Дата втрати статусу"] = df["Дата останнього рішення"].where(mask)
    with pd.ExcelWriter(name) as w:
        df.to_excel(w, sheet_name="Реєстр", index=False)
        dd.to_excel(w, sheet_name="Рішення", index=False)
    return len(df), len(dd)

rows_all, decs_all, page_no = [], [], 0
async with async_playwright() as p:
    b = await p.chromium.launch()
    page = await b.new_page()
    await page.goto(URL, wait_until="networkidle")
    while True:
        await wait_loader(page)
        await page.wait_for_selector("table tbody tr", timeout=60000)
        rows, decs = await parse_table(page)
        if not rows:
            break
        first = rows[0]["№ з/п"]
        rows_all += rows; decs_all += decs; page_no += 1
        print(len(rows_all), "компаній,", len(decs_all), "рішень (сторінка", page_no, ")")
        if page_no % 20 == 0:
            save(rows_all, decs_all, "diia_city_partial.xlsx")
        if MAX_PAGES and page_no >= MAX_PAGES:
            break

        moved = False
        for attempt in range(4):
            await wait_loader(page)
            arrow = page.locator("img[src*='arrow-pagination']").last
            if not await arrow.is_enabled():
                print("Кінець реєстру")
                break
            try:
                await arrow.click(timeout=30000)
                await page.wait_for_function(
                    """(f) => { const td = document.querySelector('table tbody tr td');
                                return td && td.innerText.trim() !== f; }""",
                    arg=first, timeout=60000)
                moved = True
                break
            except Exception as e:
                print("  повтор", attempt + 1, ":", str(e)[:80])
                await page.wait_for_timeout(5000)
        if not moved:
            break
    await b.close()

print("Готово:", save(rows_all, decs_all, "diia_city_registry_full.xlsx"))

In [ ]:
from google.colab import files
files.download("diia_city_registry_full.xlsx")